# Step efficiency: OmniLearned-small vs Transformer-small

For both **regression** and **classification**, compute how many fewer training
steps OmniLearned-small needs to reach the same validation loss that
Transformer-small achieves at the end of training.

In [1]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

WANDB_TAG = "Run_2703"

## 1. Fetch runs from wandb (same helpers as the eval notebooks)

In [2]:
from src.utils.utils import (
    get_runs_by_model_and_cap,
    get_classification_runs_by_model_and_cap,
)

reg_runs_by_model_cap = get_runs_by_model_and_cap(WANDB_TAG)
cls_runs_by_model_cap = get_classification_runs_by_model_and_cap(WANDB_TAG)

# Full-dataset (cap=-1) run names per model
reg_runs_per_model = {
    model: caps[-1]
    for model, caps in reg_runs_by_model_cap.items()
    if -1 in caps
}
cls_runs_per_model = {
    model: caps[-1]
    for model, caps in cls_runs_by_model_cap.items()
    if -1 in caps
}

print("Regression  models:", sorted(reg_runs_per_model))
print("Classification models:", sorted(cls_runs_per_model))

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /global/homes/g/gregork/.netrc.


Calling wandb API...
  Fetching runs from fcc_ml/minerva-models with tag 'Run_2703'...
  Wandb API finished: 71 run(s) found with tag 'Run_2703'.
Calling wandb API...
  Fetching runs from fcc_ml/minerva-models with tag 'Run_2703'...
  Wandb API finished: 71 run(s) found with tag 'Run_2703'.
Regression  models: ['MLP', 'OmniLearned-medium', 'OmniLearned-small', 'OmniLearned-small-rw', 'Transformer-small', 'Transformer-xsmall']
Classification models: ['MLP', 'OmniLearned-medium', 'OmniLearned-small', 'OmniLearned-small-rw', 'Transformer-small', 'Transformer-xsmall']


## 2. Download validation-loss histories

Reuse the same `get_validation_loss_history` helper defined in both eval
notebooks (duplicated inline here so this notebook is self-contained).

In [3]:
import os
import wandb


def get_validation_loss_history(run_name, project="minerva-models", with_steps=False):
    """Return (steps, losses) arrays for eval_loss logged every 1000 steps."""
    api = wandb.Api(timeout=60)
    entity = os.environ.get("WANDB_ENTITY") or getattr(api, "default_entity", None)
    if not entity:
        raise RuntimeError("WANDB_ENTITY not set and wandb default_entity unknown.")
    path = f"{entity}/{project}"
    runs = api.runs(path, filters={"displayName": run_name})
    run = next(iter(runs), None)
    if run is None:
        raise ValueError(f"Wandb run not found: {run_name!r} in {path}")
    hist = run.history()
    if "eval_loss" not in hist.columns:
        return (np.array([]), np.array([])) if with_steps else np.array([])
    mask = hist["eval_loss"].notna()
    steps = np.asarray(hist["_step"], dtype=float)[mask]
    losses = np.asarray(hist["eval_loss"].dropna(), dtype=float)
    step_int = np.round(steps).astype(int)
    keep = (step_int >= 1000) & (step_int % 1000 == 0)
    steps, losses = steps[keep], losses[keep]
    order = np.argsort(steps)
    steps, losses = steps[order], losses[order]
    _, idx = np.unique(np.round(steps).astype(int), return_index=True)
    steps = steps[np.sort(idx)]
    losses = losses[np.sort(idx)]
    if with_steps:
        return steps, losses
    return losses

## 3. Compute step efficiency

For each task (regression / classification):

1. Get the mean validation-loss curve for **Transformer-small** (across seeds).
2. Record its final (converged) validation loss.
3. Get the mean validation-loss curve for **OmniLearned-small**.
4. Find the first step at which OmniLearned-small reaches that loss.
5. Report the percentage of fewer steps relative to the total training steps
   of Transformer-small.

In [4]:
MODELS = ["OmniLearned-small", "Transformer-small"]


def build_mean_curve(run_names):
    """Return (steps_grid, mean_loss) averaged over seeds."""
    all_steps, all_losses = [], []
    for rn in run_names:
        st, lo = get_validation_loss_history(rn, with_steps=True)
        if len(st) > 0:
            all_steps.append(st)
            all_losses.append(lo)
    if not all_steps:
        return np.array([]), np.array([])
    steps_grid = np.unique(np.concatenate(all_steps)).astype(float)
    losses_aligned = np.array([
        np.interp(steps_grid, st, lo)
        for st, lo in zip(all_steps, all_losses)
    ])
    return steps_grid, np.mean(losses_aligned, axis=0)


def step_efficiency(runs_per_model, task_name):
    """Print how many fewer steps OmniLearned-small needs vs Transformer-small."""
    for m in MODELS:
        if m not in runs_per_model:
            print(f"  [skip] {m} not found in {task_name} runs")
            return None

    ts_steps, ts_loss = build_mean_curve(runs_per_model["Transformer-small"])
    ol_steps, ol_loss = build_mean_curve(runs_per_model["OmniLearned-small"])

    if len(ts_steps) == 0 or len(ol_steps) == 0:
        print(f"  [skip] empty loss history for {task_name}")
        return None

    best_idx = np.argmin(ts_loss)
    ts_final_loss = ts_loss[best_idx]
    ts_total_steps = ts_steps[best_idx]

    reached = np.where(ol_loss <= ts_final_loss)[0]
    if len(reached) == 0:
        print(f"  {task_name}: OmniLearned-small never reaches "
              f"Transformer-small final loss ({ts_final_loss:.6f})")
        return None

    ol_step_at_match = ol_steps[reached[0]]
    pct_fewer = (1 - ol_step_at_match / ts_total_steps) * 100

    print(f"  {task_name}")
    print(f"    Transformer-small final val loss : {ts_final_loss:.6f}  "
          f"(at step {ts_total_steps:.0f})")
    print(f"    OmniLearned-small reaches it at  : step {ol_step_at_match:.0f}")
    print(f"    → {pct_fewer:.1f}% fewer steps")
    return {
        "task": task_name,
        "ts_final_loss": ts_final_loss,
        "ts_total_steps": ts_total_steps,
        "ol_step_at_match": ol_step_at_match,
        "pct_fewer": pct_fewer,
    }


print("=" * 60)
reg_result = step_efficiency(reg_runs_per_model, "Regression")
print()
cls_result = step_efficiency(cls_runs_per_model, "Classification")
print("=" * 60)

  Regression
    Transformer-small final val loss : 0.032745  (at step 46000)
    OmniLearned-small reaches it at  : step 24000
    → 47.8% fewer steps

  Classification
    Transformer-small final val loss : 1.103597  (at step 35000)
    OmniLearned-small reaches it at  : step 19000
    → 45.7% fewer steps


## 4. Ready-made sentences

In [5]:
for res in [reg_result, cls_result]:
    if res is not None:
        print(
            f"The pretrained model OmniLearned-small reaches the same "
            f"validation loss as the Transformer-small in "
            f"{res['pct_fewer']:.0f}% fewer steps "
            f"({res['task'].lower()})."
        )

The pretrained model OmniLearned-small reaches the same validation loss as the Transformer-small in 48% fewer steps (regression).
The pretrained model OmniLearned-small reaches the same validation loss as the Transformer-small in 46% fewer steps (classification).
